# Comparative Trainer Benchmark — TRL GRPO vs GSPO vs DAPO

Closes the v1.0 whitepaper's biggest documented gap: the §5.6 comparative trainer table currently has only paper-reported AIME scores; the §11.3 trainer-selection decision tree is reasoned advocacy, not measured guidance. This notebook produces three-seed comparative numbers for the three trainers that fit the §11.7 customer-support task cleanly:

- **TRL GRPO** — the library baseline, delegates to Hugging Face TRL
- **GSPO** — length-normalized sequence-level ratio
- **DAPO** — Clip-Higher + dynamic sampling + overlong shaping

GEPO and VAPO are deliberately excluded:
- **GEPO** is designed for async/off-policy training; running it synchronously on a small corpus underrepresents its strengths
- **VAPO** is marked experimental in §8.1 (value-net warmup tuning still in flux); a fair comparison needs hyperparameters we haven't tuned

**Methodology mirrors §11.7 exactly:**

- Trainee: `Qwen/Qwen2.5-0.5B-Instruct` (proven headroom)
- 3 seeds: 42, 1337, 2026
- Both rubric and local LLM-judge eval (`Qwen2.5-1.5B-Instruct`)
- KL anchor enabled across all three trainers (`use_reference_model=True, beta=0.05`)
- Same number of training samples per trainer (group_size=4 × 16 scenarios × 1 epoch = 64 rollouts per seed)

**Estimated runtime:** ~45 min on Colab A100-40GB. **Cost:** ~$1.30.

**Open in Colab:** [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/stateset/stateset-agents/blob/master/notebooks/whitepaper_v1_comparative_trainers.ipynb)


## 1. Pin + install

In [ ]:
import os, subprocess, sys
PINNED_COMMIT = '8fc11b4'  # whitepaper §11.7 + reviewer-feedback v0.13.x cycle

if not os.path.exists('/content/stateset-agents'):
    subprocess.check_call(['git', 'clone', '--quiet',
        'https://github.com/stateset/stateset-agents', '/content/stateset-agents'])
subprocess.check_call(['git', '-C', '/content/stateset-agents', 'checkout', '--quiet', PINNED_COMMIT])
%cd /content/stateset-agents
print('Pinned to', subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD']).decode().strip())

In [ ]:
%pip install --quiet -e '.[training,api]'
%pip install --quiet accelerate bitsandbytes datasets
# Same Colab compatibility upgrades as the other v0.13.x notebooks.
%pip install --quiet -U transformers accelerate peft trl torchao
print('Install complete. Runtime > Restart session, then re-run from cell 1.')

## 2. Load corpus + LLM judge

In [ ]:
from stateset_agents.data import (
    load_support_scenarios, make_support_scenarios, SupportRewardComposite,
)
all_scenarios = load_support_scenarios()
train_scenarios = all_scenarios[:16]
eval_scenarios = all_scenarios[16:]
TRAINEE_MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'
JUDGE_MODEL = 'Qwen/Qwen2.5-1.5B-Instruct'
print(f'Train: {len(train_scenarios)}, Eval: {len(eval_scenarios)}')
print(f'Trainee: {TRAINEE_MODEL}\nJudge: {JUDGE_MODEL}')

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
print(f'Loading judge model: {JUDGE_MODEL}')
judge_tokenizer = AutoTokenizer.from_pretrained(JUDGE_MODEL)
judge_model = AutoModelForCausalLM.from_pretrained(
    JUDGE_MODEL, torch_dtype=torch.bfloat16, device_map='cuda',
)
judge_model.eval()
print(f'Judge loaded. VRAM after judge: {torch.cuda.memory_allocated() // (1024**2)} MB')

In [ ]:
import re
JUDGE_PROMPT = '''You are evaluating a customer service agent's response. Rate the response on a 0-10 scale.

Customer message: {query}
Expected intent: {intent}

Agent response: {response}

Rate the response on these criteria:
- Did it address the customer's intent?
- Is the tone appropriate (polite, professional)?
- Did it offer a clear next step or actionable information?

Output ONLY a single integer 0-10. Do not include any other text.

Score:'''

@torch.no_grad()
def judge_score(query, intent, response):
    prompt = JUDGE_PROMPT.format(query=query, intent=intent, response=response[:1024])
    inputs = judge_tokenizer(prompt, return_tensors='pt', truncation=True, max_length=1024).to('cuda')
    out = judge_model.generate(**inputs, max_new_tokens=8, do_sample=False, pad_token_id=judge_tokenizer.eos_token_id)
    decoded = judge_tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    m = re.search(r'\b(10|[0-9])\b', decoded)
    return min(int(m.group(1)), 10) / 10.0 if m else 0.5

## 3. Eval functions

In [ ]:
import asyncio
from stateset_agents.core import MultiTurnAgent
from stateset_agents.core.agent_config import AgentConfig
from stateset_agents.core.trajectory import ConversationTurn

def prompt_for(s):
    return ('You are a helpful customer support agent. Respond to the user warmly, '
            'address their concern directly, and confirm the next step.\n\n'
            f'User: {s.user_query}\n\nAgent:')

# Build prompt→scenario lookup so the DAPO callable reward can pull rubric context.
prompt_to_scenario = {prompt_for(s): s for s in train_scenarios}

async def evaluate(agent, scenarios):
    rubric = SupportRewardComposite()
    rubric_scores, judge_scores = [], []
    for s in scenarios:
        response = await agent.generate_response(prompt_for(s))
        turns = [ConversationTurn(role='assistant', content=response)]
        r = await rubric.compute_reward(turns, context=s.to_scenario())
        j = judge_score(s.user_query, s.intent, response)
        rubric_scores.append(r.score); judge_scores.append(j)
    n = max(len(rubric_scores), 1)
    return {'rubric_mean': sum(rubric_scores)/n, 'judge_mean': sum(judge_scores)/n}

## 4. Baseline (shared across all trainers + seeds)

Deterministic (`do_sample=False`) so the baseline number is identical regardless of seed.

In [ ]:
baseline_agent = MultiTurnAgent(AgentConfig(
    model_name=TRAINEE_MODEL,
    max_new_tokens=320, temperature=0.0, do_sample=False,
    torch_dtype='bfloat16', attn_implementation='sdpa',
))
await baseline_agent.initialize()
baseline_eval = await evaluate(baseline_agent, eval_scenarios)
print(f'Baseline rubric: {baseline_eval["rubric_mean"]:.3f}')
print(f'Baseline judge:  {baseline_eval["judge_mean"]:.3f}')
del baseline_agent
import gc; gc.collect(); torch.cuda.empty_cache()

## 5. Train + eval per (trainer, seed)

Three trainers × three seeds = nine training runs. Per-run wall clock ~3–4 min.

All three trainers use:
- `group_size=4` / `num_generations=4` (same rollout count)
- `use_reference_model=True, beta=0.05` (the §10.5 safe defaults)
- `num_epochs=1`, `learning_rate=5e-6`

The notable per-trainer divergences:
- **GSPO**: paper-default tight clip (`3e-4 / 4e-4`)
- **DAPO**: Clip-Higher asymmetric (`0.2 / 0.28`) — wider, by design
- **TRL GRPO**: PPO-default `clip_eps=0.2` symmetric

These are the trainer-defining settings, not free hyperparameters. Holding them constant would not be the same comparison.

In [ ]:
from stateset_agents.training import (
    GSPOConfig, train_with_gspo,
    train_with_trl_grpo, TRLGRPOConfig,
    train_with_dapo, DAPOConfig,
)
from stateset_agents.core import ConversationEnvironment
from stateset_agents.utils.reproducibility import set_all_seeds
import time

SEEDS = [42, 1337, 2026]
results_table = []  # rows: {trainer, seed, rubric_trained, judge_trained, ...}

# --- shared training inputs ---
env = ConversationEnvironment(
    scenarios=make_support_scenarios(train_scenarios),
    reward_fn=SupportRewardComposite(),
    max_turns=4,
)
train_queries = [
    {'prompt': prompt_for(s), 'context': {
        'must_acknowledge': list(s.must_acknowledge),
        'must_avoid': list(s.must_avoid),
        'intent': s.intent,
    }}
    for s in train_scenarios
]

# DAPO needs a callable (prompt, response) -> float. Wrap SupportRewardComposite.
_dapo_rubric = SupportRewardComposite()
async def dapo_reward_callable(prompt: str, response: str) -> float:
    s = prompt_to_scenario.get(prompt)
    if s is None:
        # If the prompt was modified during rollouts, fall back to no-rubric scoring.
        ctx = {}
    else:
        ctx = s.to_scenario()
    turns = [ConversationTurn(role='assistant', content=response)]
    r = await _dapo_rubric.compute_reward(turns, context=ctx)
    return float(r.score)
dapo_prompts = [prompt_for(s) for s in train_scenarios]

# --- per-trainer runner ---
async def run_trainer(trainer_name: str, seed: int):
    set_all_seeds(seed, deterministic_cuda=False)
    agent = MultiTurnAgent(AgentConfig(
        model_name=TRAINEE_MODEL, torch_dtype='bfloat16', attn_implementation='sdpa',
    ))
    t0 = time.time()
    if trainer_name == 'gspo':
        cfg = GSPOConfig(
            model_name=TRAINEE_MODEL, num_generations=4,
            clip_range_left=3e-4, clip_range_right=4e-4,
            learning_rate=5e-6, max_prompt_length=512, max_completion_length=320,
            use_lora=True, lora_r=16, lora_alpha=32,
            gradient_checkpointing=False, num_epochs=1, warmup_ratio=0.1,
            use_reference_model=True, beta=0.05,
            output_dir=f'/content/compare_gspo_seed{seed}',
        )
        await train_with_gspo(config=cfg, agent=agent, environment=env,
                              reward_model=env.reward_fn, train_queries=train_queries)
    elif trainer_name == 'trl_grpo':
        cfg = TRLGRPOConfig(
            model_name=TRAINEE_MODEL, num_generations=4,
            beta=0.05,                     # KL anchor — same as GSPO/DAPO
            use_reference_model=True,
            learning_rate=5e-6, max_prompt_length=512, max_completion_length=320,
            use_lora=True, lora_r=16, lora_alpha=32,
            gradient_checkpointing=False, num_epochs=1, warmup_ratio=0.1,
            output_dir=f'/content/compare_trlgrpo_seed{seed}',
        )
        await train_with_trl_grpo(config=cfg, agent=agent, environment=env,
                                  reward_model=env.reward_fn)
    elif trainer_name == 'dapo':
        cfg = DAPOConfig(
            model_name=TRAINEE_MODEL,
            group_size=4,                    # match GSPO's rollout count
            prompt_batch_size=16,            # corpus size — DAPO defaults to 512 which would never finish on a 16-scenario corpus
            num_gradient_updates=4,          # default 16 is too many for 16 scenarios
            clip_eps_low=0.2, clip_eps_high=0.28,  # Clip-Higher, the trainer's defining mechanism
            learning_rate=5e-6, lr_scheduler_type='constant',
            use_lora=True, lora_r=16, lora_alpha=32,
            gradient_checkpointing=False, num_epochs=1, warmup_ratio=0.1,
            use_reference_model=True, beta=0.05,
            output_dir=f'/content/compare_dapo_seed{seed}',
        )
        # DAPO entrypoint takes model_name + reward_fn + train_prompts (not (config, agent, env))
        _, _, _ = await train_with_dapo(
            model_name=TRAINEE_MODEL,
            reward_fn=dapo_reward_callable,
            train_prompts=dapo_prompts,
            config=cfg,
            output_dir=cfg.output_dir,
        )
        # DAPO returns (model, tokenizer, metrics) — wire it into the agent for eval
        # The trained model/tokenizer aren't always returned to a usable agent here;
        # the cleanest workaround is to re-initialize the agent from the output_dir.
        agent = MultiTurnAgent(AgentConfig(
            model_name=cfg.output_dir if os.path.exists(os.path.join(cfg.output_dir, 'adapter_model.safetensors')) else TRAINEE_MODEL,
            torch_dtype='bfloat16', attn_implementation='sdpa',
        ))
        await agent.initialize()
    else:
        raise ValueError(trainer_name)
    wall_clock = time.time() - t0
    metrics = await evaluate(agent, eval_scenarios)
    del agent; gc.collect(); torch.cuda.empty_cache()
    return {**metrics, 'wall_clock_seconds': wall_clock}

# --- loop ---
for trainer_name in ['trl_grpo', 'gspo', 'dapo']:
    print(f'\n========== Trainer: {trainer_name} ==========')
    for seed in SEEDS:
        print(f'  seed {seed} ...', end=' ', flush=True)
        try:
            m = await run_trainer(trainer_name, seed)
            row = {
                'trainer': trainer_name, 'seed': seed,
                'rubric_trained': m['rubric_mean'],
                'judge_trained': m['judge_mean'],
                'rubric_improvement': m['rubric_mean'] - baseline_eval['rubric_mean'],
                'judge_improvement': m['judge_mean'] - baseline_eval['judge_mean'],
                'wall_clock_seconds': m['wall_clock_seconds'],
                'error': None,
            }
        except Exception as e:
            row = {
                'trainer': trainer_name, 'seed': seed,
                'rubric_trained': None, 'judge_trained': None,
                'rubric_improvement': None, 'judge_improvement': None,
                'wall_clock_seconds': None,
                'error': f'{type(e).__name__}: {str(e)[:200]}',
            }
            print(f'FAILED: {row["error"]}')
        else:
            print(f'rubric={row["rubric_improvement"]:+.3f}  judge={row["judge_improvement"]:+.3f}  ({row["wall_clock_seconds"]:.0f}s)')
        results_table.append(row)


## 6. Aggregate + verdict

In [ ]:
import statistics

def agg(rows, metric_key):
    vals = [r[metric_key] for r in rows if r[metric_key] is not None]
    if not vals: return {'n': 0, 'mean': None, 'stdev': None, 'agreement': False}
    return {
        'n': len(vals),
        'mean': statistics.mean(vals),
        'stdev': statistics.stdev(vals) if len(vals) > 1 else 0.0,
        'agreement': all(v > 0 for v in vals) or all(v < 0 for v in vals),
    }

per_trainer = {}
for trainer_name in ['trl_grpo', 'gspo', 'dapo']:
    rows = [r for r in results_table if r['trainer'] == trainer_name]
    per_trainer[trainer_name] = {
        'rubric': agg(rows, 'rubric_improvement'),
        'judge':  agg(rows, 'judge_improvement'),
    }

print('=' * 80)
print(f'Baseline: rubric={baseline_eval["rubric_mean"]:.3f}  judge={baseline_eval["judge_mean"]:.3f}')
print('=' * 80)
print(f'{"trainer":<10}  {"metric":<6}  {"mean Δ":>9}  {"σ":>7}  {"3-seed agreement":>16}')
print('-' * 80)
for trainer_name, stats in per_trainer.items():
    for metric_key in ['rubric', 'judge']:
        s = stats[metric_key]
        mean_str = f'{s["mean"]:+.3f}' if s['mean'] is not None else '   -  '
        std_str  = f'{s["stdev"]:.3f}'  if s['stdev'] is not None else '  -  '
        print(f'{trainer_name:<10}  {metric_key:<6}  {mean_str:>9}  {std_str:>7}  {str(s["agreement"]):>16}')

# Whitepaper publication gate (per SCHEMA.md): judge improvement > 0.03 with 3-seed agreement.
print()
print('Per-trainer publication gate (judge Δ > 0.03 ∧ 3-seed agreement):')
for trainer_name, stats in per_trainer.items():
    s = stats['judge']
    passed = s['mean'] is not None and s['mean'] > 0.03 and s['agreement']
    print(f'  {trainer_name:<10}: {"PASS ✅" if passed else "FAIL ❌"}')

## 7. Save schema-compliant comparative JSON

In [ ]:
import json
from datetime import datetime, timezone
from pathlib import Path

result = {
    'protocol': 'whitepaper_v1_comparative_trainers',
    'task': 'customer_support',
    'model': TRAINEE_MODEL,
    'seeds': SEEDS,
    'judge_model': JUDGE_MODEL,
    'commit': PINNED_COMMIT,
    'timestamp': datetime.now(timezone.utc).isoformat(),
    'baseline': baseline_eval,
    'rows': results_table,
    'per_trainer_aggregate': per_trainer,
    'hardware': {
        'gpu': torch.cuda.get_device_name(0),
        'cuda': torch.version.cuda,
        'peak_vram_mb': torch.cuda.max_memory_allocated() // (1024**2),
    },
}
out = Path('/content/whitepaper_v1_comparative_trainers.json')
out.write_text(json.dumps(result, indent=2, default=str))
print(json.dumps(result, indent=2, default=str))
print(f'\nSaved to: {out}')

## 8. What this fills in for the whitepaper

The §5.6 comparative trainer summary in the whitepaper currently lists paper-reported AIME scores (DAPO 50/60, VAPO 60.4) but no first-party numbers comparing them on a shared task. This notebook fills that gap with first-party measurements on the §11.7 customer-support protocol.

The §11.3 decision tree ("first run on a new task → TRL GRPO; long outputs → GSPO; reasoning task with sparse reward → DAPO") becomes a *measured* recommendation only when the comparative numbers are in. The values produced by this notebook should be committed to `benchmark_results/whitepaper_v1/comparative_trainers_qwen25_05b.json` and cited from §5.6 and §11.3.
